# Geographic Scalars for Temperature (2m)

This tutorial shows how loss scalars on the ERA5 0.25° grid are generated from **population density**. The same scalar field is intended for `2m_temperature`. Wind u/v and SSRD use separate capacity-weighted fields.

## Data sources

1. **Gridded Population of the World (GPWv4)** population density (15 arc-minute) — ESRI ASCII `.asc` raster.
2. **Natural Earth 50m** — country polygons used to build buffered Europe and Germany masks.
3. **ERA5 0.25° grid** — 721 latitudes in `[-90, 90]` and 1440 longitudes in `[-180, 179.75]`.

### Getting the population data

1. The density data is from US GHG Center : https://earth.gov/ghgcenter/data-catalog/sedac-popdensity-yeargrid5yr-v4.11
1. Obtain GPWv4 population density (15 minute), revision 11, year 2020 (or the release used by this repo).
2. Save the ESRI ASCII file next to this notebook as:
   `gpw_v4_population_density_rev11_2020_15_min.asc`

> Population density (people / km²) is interpolated onto the ERA5 grid with nearest-neighbour resampling after NODATA (-9999) is set to 0.

> Center For International Earth Science Information Network-CIESIN-Columbia University. (2017). Gridded Population of the World, Version 4 (GPWv4): Population Density, Revision 11 (Version 4.11)[gpw_v4_population_density_rev11_2020_15_min.asc], Palisades, NY: Socioeconomic Data and Applications Center (SEDAC). https://doi.org/10.7927/H49C6VHW Date Accessed: [21/08/2026]


## Setup

Install the packages needed to run this notebook by itself.


In [ ]:
%pip install -q numpy scipy geopandas shapely matplotlib cartopy requests rioxarray xarray pandas


In [1]:
import io
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import rioxarray
import scipy.ndimage as ndimage
from matplotlib.colors import LogNorm
from shapely.geometry import Point
from shapely.prepared import prep

ASC_NAME = "gpw_v4_population_density_rev11_2020_15_min.asc"
CANDIDATES = [
    Path.cwd(),
    Path.cwd() / "notebooks",
    Path("/root/Zeus/notebooks"),
]
NOTEBOOK_DIR = next(
    (p for p in CANDIDATES if (p / ASC_NAME).exists()),
    Path("/root/Zeus/notebooks"),
)
EXPORT_DIR = NOTEBOOK_DIR
WEIGHTS_DIR = Path("/root/Zeus/zeus/data/weights")
print("Notebook / data directory:", NOTEBOOK_DIR.resolve())
print("Population ASC:", (NOTEBOOK_DIR / ASC_NAME).exists())


Notebook / data directory: /root/Zeus/notebooks
Population ASC: True


## 1. ERA5 grid, region masks, and population load

Build the ERA5 0.25° grid, buffered Europe / Germany masks (2°), then load and align the GPW population density raster onto that grid.


In [2]:
GEOJSON_URL = (
    "https://raw.githubusercontent.com/nvkelso/natural-earth-vector/master/"
    "geojson/ne_50m_admin_0_countries.geojson"
)

N_LAT = 721
N_LON = 1440
GRID_RES = 0.25
BUFFER_DEG = 2.0

EUROPE_LAT_MIN, EUROPE_LAT_MAX = 27.5, 72.25
EUROPE_LON_MIN, EUROPE_LON_MAX = -25.50, 45.25
GERMANY_LAT_MIN, GERMANY_LAT_MAX = 47.0, 56.25
GERMANY_LON_MIN, GERMANY_LON_MAX = 5.25, 15.25


def make_era5_grid():
    """0.25° grid: latitude south→north, longitude in [-180, 180)."""
    lats = np.linspace(-90.0, 90.0, N_LAT)
    lons = np.linspace(-180.0, 179.75, N_LON)
    return lats, lons


def fetch_country_geometries():
    headers = {"User-Agent": "Mozilla/5.0 (X11; Linux x86_64)"}
    response = requests.get(GEOJSON_URL, headers=headers, timeout=30)
    response.raise_for_status()
    return gpd.read_file(io.BytesIO(response.content))


def generate_region_masks(lats, lons, buffer_deg=BUFFER_DEG):
    gdf = fetch_country_geometries()

    germany_geom = gdf[gdf["ADMIN"] == "Germany"].geometry.union_all()
    europe_geom = gdf[gdf["CONTINENT"] == "Europe"].geometry.union_all()
    russia_geom = gdf[gdf["ADMIN"] == "Russia"].geometry.union_all()
    europe_geom = europe_geom.difference(russia_geom)

    europe_buffered = europe_geom.buffer(buffer_deg)
    germany_buffered = germany_geom.buffer(buffer_deg)

    prep_eu = prep(europe_buffered)
    prep_de = prep(germany_buffered)

    grid_lon, grid_lat = np.meshgrid(lons, lats)
    flat_lons = grid_lon.flatten()
    flat_lats = grid_lat.flatten()

    de_mask_flat = np.zeros(len(flat_lats), dtype=bool)
    eu_mask_flat = np.zeros(len(flat_lats), dtype=bool)

    eu_candidate_idx = np.where(
        (flat_lats >= EUROPE_LAT_MIN)
        & (flat_lats < EUROPE_LAT_MAX)
        & (flat_lons >= EUROPE_LON_MIN)
        & (flat_lons < EUROPE_LON_MAX)
    )[0]

    de_candidate_idx = np.where(
        (flat_lats >= GERMANY_LAT_MIN)
        & (flat_lats < GERMANY_LAT_MAX)
        & (flat_lons >= GERMANY_LON_MIN)
        & (flat_lons < GERMANY_LON_MAX)
    )[0]

    for idx in eu_candidate_idx:
        if prep_eu.contains(Point(flat_lons[idx], flat_lats[idx])):
            eu_mask_flat[idx] = True

    for idx in de_candidate_idx:
        if prep_de.contains(Point(flat_lons[idx], flat_lats[idx])):
            de_mask_flat[idx] = True

    eu_mask_flat[de_mask_flat] = False

    de_mask = de_mask_flat.reshape((len(lats), len(lons)))
    eu_mask = eu_mask_flat.reshape((len(lats), len(lons)))
    row_mask = ~(de_mask | eu_mask)
    return de_mask, eu_mask, row_mask


def load_population_grid(asc_path, lats, lons):
    """Load ESRI ASCII population density and align to the ERA5 lat/lon axes."""
    pop_da = rioxarray.open_rasterio(asc_path).squeeze()
    pop_da = pop_da.where(pop_da != -9999, 0.0)
    pop_da = pop_da.rename({"y": "latitude", "x": "longitude"})
    aligned_pop = pop_da.interp(
        latitude=lats,
        longitude=lons,
        method="nearest",
        kwargs={"fill_value": 0.0},
    )
    return np.nan_to_num(aligned_pop.values, nan=0.0)


lats, lons = make_era5_grid()
de_mask, eu_mask, row_mask = generate_region_masks(lats, lons)
pop_grid = load_population_grid(NOTEBOOK_DIR / ASC_NAME, lats, lons)

assert pop_grid.shape == (N_LAT, N_LON), pop_grid.shape
assert np.isfinite(pop_grid).all()
assert float(pop_grid.min()) >= 0.0

print(f"Grid: {len(lats)} x {len(lons)} (res={GRID_RES}°)")
print(f"Germany cells: {int(de_mask.sum())}")
print(f"Europe (ex-Germany) cells: {int(eu_mask.sum())}")
print(f"Rest of world cells: {int(row_mask.sum())}")
print(
    f"Population: min={pop_grid.min():.4g} max={pop_grid.max():.4g} "
    f"mean={pop_grid.mean():.4g} occupied={(pop_grid > 0).sum()}"
)


Grid: 721 x 1440 (res=0.25°)
Germany cells: 1447
Europe (ex-Germany) cells: 23483
Rest of world cells: 1013310
Population: min=0 max=2.982e+04 mean=14.77 occupied=229218


## 2. Population → temperature scalars

Pipeline:

1. Start from the **population density** grid already on the ERA5 mesh (no plant binning).
2. **Smooth** with a Gaussian filter (`sigma=1.5` grid cells, `mode="wrap"`).
3. **Assign regional budgets**: Germany **0.40**, rest of Europe **0.40**, rest of world **0.20**.

For rest of world, every cell gets `0.20 / n_row`.

For Germany and Europe, each cell first receives that ROW baseline. The leftover regional budget is distributed **in proportion to smoothed population**:

```text
scalar[cell] = row_baseline + signal_budget * (smoothed_population[cell] / sum(smoothed_population in region))
```

Finally rescale so the global minimum is **1** (`scalars / scalars.min()`).


In [3]:
def compute_global_scalar_matrix(
    raw_pop_grid, de_mask, eu_mask, row_mask, sigma_grid_cells=1.5
):
    if raw_pop_grid.ndim != 2:
        raise ValueError(f"raw_pop_grid must be 2D, got shape {raw_pop_grid.shape}")
    if not (raw_pop_grid.shape == de_mask.shape == eu_mask.shape == row_mask.shape):
        raise ValueError(
            f"pop/mask shape mismatch: pop={raw_pop_grid.shape} "
            f"de={de_mask.shape} eu={eu_mask.shape} row={row_mask.shape}"
        )

    smoothed_grid = ndimage.gaussian_filter(
        raw_pop_grid, sigma=sigma_grid_cells, mode="wrap"
    )

    num_row_cells = np.count_nonzero(row_mask)
    row_cell_value = 0.20 / num_row_cells

    scalars = np.zeros(raw_pop_grid.shape, dtype=np.float64)
    scalars[row_mask] = row_cell_value

    def _assign_regional_scalars(mask, regional_total):
        count = int(np.count_nonzero(mask))
        if count == 0:
            return
        baseline_total = count * row_cell_value
        signal_budget = regional_total - baseline_total
        signal_data = smoothed_grid[mask]
        signal_sum = float(signal_data.sum())
        if signal_budget <= 0.0 or signal_sum <= 0.0:
            scalars[mask] = regional_total / count
            return
        scalars[mask] = row_cell_value + signal_budget * (signal_data / signal_sum)

    _assign_regional_scalars(de_mask, 0.40)
    _assign_regional_scalars(eu_mask, 0.40)
    return scalars


def normalize_scalars_min_one(scalars):
    scalar_min = float(np.min(scalars))
    if scalar_min <= 0.0:
        raise ValueError(f"Cannot normalize non-positive scalars, min={scalar_min}")
    return scalars / scalar_min


scalars_raw = compute_global_scalar_matrix(pop_grid, de_mask, eu_mask, row_mask)
scalars = normalize_scalars_min_one(scalars_raw)

print(f"Unnormalized sum (should be ~1): {scalars_raw.sum():.6f}")
print(f"Normalized min/max: {scalars.min():.4f} / {scalars.max():.4f}")
print(
    f"DE/EU/ROW sums: "
    f"{scalars_raw[de_mask].sum():.4f} / "
    f"{scalars_raw[eu_mask].sum():.4f} / "
    f"{scalars_raw[row_mask].sum():.4f}"
)


Unnormalized sum (should be ~1): 1.000000
Normalized min/max: 1.0000 / 7245.5936
DE/EU/ROW sums: 0.4000 / 0.4000 / 0.2000


## 3. Region statistics

Stats use the Europe / Germany lat–lon boxes. Europe here means the Europe box **excluding** the Germany box; rest of world is everything outside the Europe box.


In [4]:
def _bbox_mask(lats, lons, lat_min, lat_max, lon_min, lon_max):
    lat_sel = (lats >= lat_min) & (lats < lat_max)
    lon_sel = (lons >= lon_min) & (lons < lon_max)
    return lat_sel[:, None] & lon_sel[None, :]


def print_region_stats(values, lats, lons, label):
    europe_bbox = _bbox_mask(
        lats, lons, EUROPE_LAT_MIN, EUROPE_LAT_MAX, EUROPE_LON_MIN, EUROPE_LON_MAX
    )
    germany_mask = _bbox_mask(
        lats, lons, GERMANY_LAT_MIN, GERMANY_LAT_MAX, GERMANY_LON_MIN, GERMANY_LON_MAX
    )
    regions = {
        "Europe": europe_bbox & ~germany_mask,
        "Germany": germany_mask,
        "Rest of world": ~europe_bbox,
    }
    print(f"\n{label}")
    for name, mask in regions.items():
        vals = values[mask]
        print(
            f"  {name}: cells={int(mask.sum())} "
            f"min={float(vals.min()):.6e} "
            f"max={float(vals.max()):.6e} "
            f"mean={float(vals.mean()):.6e} "
            f"sum={float(vals.sum()):.6e}"
        )
    print(f"  Total sum: {float(values.sum()):.6e}")


print_region_stats(scalars_raw, lats, lons, "Temperature scalar stats (unnormalized)")
print_region_stats(scalars, lats, lons, "Temperature scalar stats (normalized min=1)")



Temperature scalar stats (unnormalized)
  Europe: cells=49177 min=1.973730e-07 max=7.985220e-04 mean=8.233575e-06 sum=4.049025e-01
  Germany: cells=1480 min=1.973730e-07 max=1.430084e-03 mean=2.703887e-04 sum=4.001752e-01
  Rest of world: cells=987583 min=1.973730e-07 max=1.973730e-07 mean=1.973730e-07 sum=1.949222e-01
  Total sum: 1.000000e+00

Temperature scalar stats (normalized min=1)
  Europe: cells=49177 min=1.000000e+00 max=4.045752e+03 mean=4.171582e+01 sum=2.051459e+06
  Germany: cells=1480 min=1.000000e+00 max=7.245594e+03 mean=1.369938e+03 sum=2.027508e+06
  Rest of world: cells=987583 min=1.000000e+00 max=1.000000e+00 mean=1.000000e+00 sum=9.875830e+05
  Total sum: 5.066550e+06


## 4. Map of normalized scalars

Cartopy layout matches the wind/solar geographic-scalar maps. Values are continuous (population-weighted), so a **log** color scale is used.


In [6]:
plt.style.use("default")
plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 11,
        "axes.labelsize": 12,
        "axes.titlesize": 14,
        "figure.titlesize": 16,
    }
)

fig = plt.figure(figsize=(12, 6))
ax = plt.axes(projection=ccrs.PlateCarree(central_longitude=0))
ax.set_extent([-80, 80, 0, 85], crs=ccrs.PlateCarree())

ax.add_feature(
    cfeature.NaturalEarthFeature(
        "physical", "land", "50m", edgecolor="face", facecolor="#f4f6f8"
    ),
    zorder=0,
)
ax.add_feature(
    cfeature.NaturalEarthFeature(
        "physical", "ocean", "50m", edgecolor="face", facecolor="#ffffff"
    ),
    zorder=0,
)
ax.add_feature(
    cfeature.BORDERS.with_scale("50m"),
    linewidth=0.3,
    edgecolor="#999999",
    linestyle="--",
    zorder=0,
)
ax.coastlines(resolution="50m", linewidth=0.5, color="#333333", zorder=1)

norm = LogNorm(vmin=float(scalars.min()), vmax=float(scalars.max()))
img = ax.pcolormesh(
    lons,
    lats,
    scalars,
    transform=ccrs.PlateCarree(),
    cmap="YlOrRd",
    norm=norm,
    shading="auto",
    alpha=0.75,
    zorder=2,
)

cbar = plt.colorbar(img, ax=ax, orientation="vertical", pad=0.02, fraction=0.035, shrink=0.85)
cbar.set_label("Normalized scalar (log scale)", fontsize=11)

gl = ax.gridlines(
    draw_labels=True, linewidth=0.5, color="gray", alpha=0.3, linestyle=":"
)
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = {"size": 9, "color": "#333333"}
gl.ylabel_style = {"size": 9, "color": "#333333"}

plt.title(
    "Geographic Scalars for 2m Temperature (population-weighted, min = 1)",
    pad=20,
    weight="bold",
)

# Avoid bbox_inches="tight" with cartopy gridline labels (colorbar-only PNG bug).
fig.canvas.draw()
png_path = EXPORT_DIR / "new_temperature_geographic_scalars.png"
plt.savefig(png_path, dpi=300, pad_inches=0.2)
print(f"Exported to {png_path}")
plt.close()


Exported to /root/Zeus/notebooks/new_temperature_geographic_scalars.png


In [7]:
# Save normalized scalars with lat/lon axes for Zeus scoring
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
npz_path = WEIGHTS_DIR / "new_temperature_scalars.npz"
np.savez(npz_path, scalars=scalars, lats=lats, lons=lons)
print(f"Saved {npz_path}")
print(f"  scalars={scalars.shape} lats={lats.shape} lons={lons.shape}")
print(f"  lat [{lats[0]:.2f}, {lats[-1]:.2f}] lon [{lons[0]:.2f}, {lons[-1]:.2f}]")


Saved /root/Zeus/zeus/data/weights/new_temperature_scalars.npz
  scalars=(721, 1440) lats=(721,) lons=(1440,)
  lat [-90.00, 90.00] lon [-180.00, 179.75]


> Scoring mean-normalizes geographic weights after combining them with latitude weights (`zeus/validator/metrics.py` → `custom_rmse()`), so either the raw or min=1 field can be used as input.
